# Insight 1 — Crescimento Limitado pela Logística

**Tech Challenge Fase 1 — POSTECH DTAT**

**Pergunta de negócio:** O crescimento de pedidos e receita do Olist está sendo limitado pela logística?

**Dataset:** Brazilian E-Commerce Public Dataset by Olist (~100 mil pedidos, 2016–2018)

## 0. Setup

Importação de bibliotecas e configuração do ambiente.

In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

# Importar modulos do projeto
from src.config import DATA_DIR
from src.pipeline_limpeza import (
    carregar_datasets, filtrar_pedidos_entregues,
    converter_datas, calcular_tempo_entrega, remover_outliers_entrega,
)
from src.pipeline_joins import (
    juntar_orders_customers, juntar_com_payments, mapear_regioes, MAPA_UF_REGIAO,
)
from src.analisador_tendencia import (
    calcular_tendencia_linear, calcular_pearson,
    classificar_velocidade_entrega, calcular_ticket_medio_por_grupo,
)
from src.gerador_graficos import (
    grafico_evolucao_mensal, grafico_receita_por_estado,
    grafico_scatter_entrega_receita, grafico_evolucao_por_regiao,
    grafico_ticket_medio_comparativo,
)

print(f'DATA_DIR: {DATA_DIR}')
print('Setup concluido!')

## 1. Carregamento dos Dados

Carregamento dos 3 arquivos CSV do dataset Olist.

In [ ]:
df_orders, df_customers, df_payments = carregar_datasets(DATA_DIR)

In [ ]:
print('=== Orders ===')
display(df_orders.head(3))
print('\n=== Customers ===')
display(df_customers.head(3))
print('\n=== Payments ===')
display(df_payments.head(3))

## 2. Limpeza dos Dados

Pipeline de limpeza: filtragem, conversão de datas, cálculo do tempo de entrega e remoção de outliers.

### 2.1 Filtragem — Apenas pedidos entregues

Filtramos apenas pedidos com `order_status == 'delivered'`.

In [ ]:
registros_brutos = len(df_orders)
print(f'Registros brutos: {registros_brutos}')

df_orders = filtrar_pedidos_entregues(df_orders)
registros_apos_filtragem = len(df_orders)

### 2.2 Conversão de datas

Convertemos as colunas de data para `datetime` e removemos registros com datas inválidas.

In [ ]:
df_orders = converter_datas(df_orders)
registros_apos_conversao = len(df_orders)

### 2.3 Cálculo do tempo de entrega

`tempo_entrega` = diferença em dias entre entrega e compra.

In [ ]:
df_orders = calcular_tempo_entrega(df_orders)

### 2.4 Remoção de outliers

Removemos entregas com tempo negativo ou superior a 60 dias.

In [ ]:
df_orders = remover_outliers_entrega(df_orders)
registros_apos_outliers = len(df_orders)

## 3. Consolidação dos Dados (Joins)

Junção das tabelas de pedidos, clientes e pagamentos.

### 3.1 Join orders + customers

In [ ]:
df = juntar_orders_customers(df_orders, df_customers)

### 3.2 Join com payments (pré-agregado)

Pré-agregamos pagamentos por `order_id` para evitar duplicação.

In [ ]:
df = juntar_com_payments(df, df_payments)
registros_apos_joins = len(df)

### 3.3 Mapeamento de regiões

In [ ]:
df = mapear_regioes(df)
print(f'\nDataFrame consolidado: {df.shape[0]} registros, {df.shape[1]} colunas')
display(df.head())

## 4. Governança e Qualidade dos Dados

Resumo de todas as decisões de limpeza e transformação.

In [ ]:
print('=' * 60)
print('RESUMO DO PIPELINE DE DADOS')
print('=' * 60)
print(f'Registros brutos (orders):          {registros_brutos:>8,}')
print(f'Apos filtragem (delivered):         {registros_apos_filtragem:>8,}')
print(f'Apos conversao de datas:            {registros_apos_conversao:>8,}')
print(f'Apos remocao de outliers:           {registros_apos_outliers:>8,}')
print(f'Apos joins (consolidado final):     {registros_apos_joins:>8,}')
print('=' * 60)

### Decisões documentadas

| Decisão | Justificativa |
|---|---|
| Filtrar apenas `order_status == 'delivered'` | Análise foca em entregas concluídas |
| `tempo_entrega = delivered_date - purchase_date` (dias) | Tempo total percebido pelo cliente |
| Remover `tempo_entrega < 0` | Erro nos dados |
| Remover `tempo_entrega > 60` | Outliers extremos |
| Pré-agregar payments por `order_id` | Evitar duplicação de linhas |

## 5. Gráfico 1 — Evolução Mensal de Pedidos

In [ ]:
fig = grafico_evolucao_mensal(df, mostrar_tendencia=True)
plt.show()

### Análise

O volume de pedidos saiu de **262 em out/2016** para um pico de **7.261 em nov/2017**, com tendência de crescimento de **+333 pedidos/mês**. Após o pico, houve desaceleração — o último mês registrado (ago/2018) teve 6.351 pedidos. A linha de tendência ascendente confirma o crescimento geral, mas a desaceleração nos meses finais sugere que fatores operacionais — como a capacidade logística — podem estar limitando o potencial de expansão.

## 6. Gráfico 2 — Receita Total por Estado

In [ ]:
fig = grafico_receita_por_estado(df)
plt.show()

### Análise

A receita total de **R$ 15,36 milhões** está fortemente concentrada: **SP (37,5%)**, **RJ (13,3%)** e **MG (11,8%)** somam **62,6%** do total. Os 24 estados restantes dividem apenas 37,4%. Estados do Norte (1,8% dos pedidos) e Nordeste (9,3%) contribuem com parcelas significativamente menores, indicando potencial de receita subexplorado — possivelmente limitado por frete alto e prazo elevado nessas regiões.

## 7. Gráfico 3 — Tempo Médio de Entrega vs Receita por Estado

In [ ]:
fig = grafico_scatter_entrega_receita(df)
plt.show()

### Análise

A correlação de Pearson entre tempo médio de entrega e receita por estado é **r = −0,643 (p = 0,0003)** — correlação negativa moderada-forte e estatisticamente significativa. Estados com entregas mais rápidas geram mais receita. SP lidera com tempo médio de ~10 dias e maior receita. O Norte, com média de 21,4 dias, tem a menor participação. Isso evidencia que a logística é fator determinante na geração de receita.

## 8. Enriquecimento — Evolução Mensal por Região

In [ ]:
fig = grafico_evolucao_por_regiao(df)
plt.show()

### Análise

O Sudeste domina com **68,7%** dos pedidos (tempo médio de entrega: 10,1 dias). O Sul vem em segundo com 14,3% (13,4 dias), seguido por Nordeste 9,3% (18,9 dias), Centro-Oeste 5,8% (14,4 dias) e Norte 1,8% (21,4 dias). A diferença crescente entre as curvas sugere que a infraestrutura logística concentrada no Sudeste favorece o crescimento nessa região, enquanto regiões mais distantes ficam para trás.

## 9. Enriquecimento — Ticket Médio: Entrega Rápida vs Lenta

In [ ]:
# Agregar por UF
df_por_uf = df.groupby('customer_state').agg(
    tempo_medio=('tempo_entrega', 'mean'),
).reset_index()

# Classificar UFs
df_rapida, df_lenta = classificar_velocidade_entrega(df_por_uf)

# Calcular ticket medio
ticket_r, ticket_l, diff = calcular_ticket_medio_por_grupo(df, df_rapida, df_lenta)

# Grafico comparativo
fig = grafico_ticket_medio_comparativo(ticket_r, ticket_l, diff)
plt.show()

### Análise

Estados com entrega rápida (< 18 dias, 13 UFs) apresentam ticket médio de **R$ 154,95**, enquanto estados com entrega lenta (≥ 18 dias, 14 UFs) têm ticket médio de **R$ 206,21** — uma diferença de **−24,9%**. Isso indica que, embora regiões distantes paguem mais por pedido (possivelmente pelo frete embutido), o volume muito menor de pedidos nessas regiões resulta em receita total inferior. A performance logística impacta diretamente o comportamento de compra.

## 10. Exportação CSV para Power BI

Exportamos o DataFrame consolidado e tabelas agregadas para importação no Power BI.

In [ ]:
import os

# Diretorio de saida
OUTPUT_DIR = 'output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. DataFrame consolidado (base principal pro Power BI)
colunas_export = [
    'order_id', 'customer_id', 'order_purchase_timestamp',
    'order_delivered_customer_date', 'order_estimated_delivery_date',
    'tempo_entrega', 'customer_city', 'customer_state',
    'payment_value', 'regiao',
]
df[colunas_export].to_csv(
    os.path.join(OUTPUT_DIR, 'insight1_consolidado.csv'),
    index=False, encoding='utf-8-sig',
)
print(f'insight1_consolidado.csv: {len(df)} registros exportados')

# 2. Agregado mensal (pra grafico de evolucao)
df_mensal = df.copy()
df_mensal['ano_mes'] = df_mensal['order_purchase_timestamp'].dt.to_period('M').astype(str)
agg_mensal = df_mensal.groupby('ano_mes').agg(
    qtd_pedidos=('order_id', 'nunique'),
    receita_total=('payment_value', 'sum'),
).reset_index()
agg_mensal.to_csv(
    os.path.join(OUTPUT_DIR, 'insight1_mensal.csv'),
    index=False, encoding='utf-8-sig',
)
print(f'insight1_mensal.csv: {len(agg_mensal)} meses exportados')

# 3. Agregado por UF (pra scatter e barras)
agg_uf = df.groupby(['customer_state', 'regiao']).agg(
    qtd_pedidos=('order_id', 'nunique'),
    receita_total=('payment_value', 'sum'),
    tempo_medio_entrega=('tempo_entrega', 'mean'),
    ticket_medio=('payment_value', 'mean'),
).reset_index()
agg_uf.to_csv(
    os.path.join(OUTPUT_DIR, 'insight1_por_uf.csv'),
    index=False, encoding='utf-8-sig',
)
print(f'insight1_por_uf.csv: {len(agg_uf)} UFs exportadas')

# 4. Agregado por regiao mensal (pra evolucao por regiao)
agg_regiao = df_mensal.groupby(['ano_mes', 'regiao']).agg(
    qtd_pedidos=('order_id', 'nunique'),
    receita_total=('payment_value', 'sum'),
).reset_index()
agg_regiao.to_csv(
    os.path.join(OUTPUT_DIR, 'insight1_regiao_mensal.csv'),
    index=False, encoding='utf-8-sig',
)
print(f'insight1_regiao_mensal.csv: {len(agg_regiao)} registros exportados')

print(f'\nTodos os CSVs exportados para ./{OUTPUT_DIR}/')

## 11. Conclusão e Recomendação

### Síntese dos achados

1. **O crescimento de pedidos desacelerou** — de 262 (out/2016) para pico de 7.261 (nov/2017), com tendência de +333 pedidos/mês, mas desaceleração nos meses finais.
2. **A receita está concentrada em SP–RJ–MG** (62,6% de R$ 15,36 mi), com 24 estados dividindo apenas 37,4%.
3. **Correlação negativa forte (Pearson r = −0,643, p < 0,001)** entre tempo de entrega e receita: logística é o principal gargalo.
4. **O Sudeste concentra 68,7% dos pedidos** (tempo médio 10,1 dias) vs Norte com 1,8% (21,4 dias).
5. **O ticket médio é R$ 154,95 (entrega rápida) vs R$ 206,21 (entrega lenta)** — regiões distantes pagam mais por pedido, mas compram muito menos.

### Recomendação para o investidor

> **A demanda existe e cresceu 24x em 2 anos, mas a capacidade logística não acompanhou.** A correlação negativa (r = −0,643) entre tempo de entrega e receita mostra que estados fora do eixo SP–RJ–MG possuem potencial de receita subexplorado. Investir em centros de distribuição regionais (especialmente Nordeste e Norte), incentivo a sellers com SLA rápido e otimização de rotas é a maior alavanca de crescimento de receita para o Olist.